# 🚀 Soil GSD: Multi-Output LightGBM Baseline

This notebook provides a complete end-to-end Machine Learning baseline using **LightGBM** and traditional **Computer Vision feature extraction**. 

### Approach:
1. **Fast Image Mapping:** Pre-scan image directories into a hash map to avoid slow I/O operations during feature engineering.
2. **Feature Engineering:** Extract color statistics, spatial dimensions, and **Edge Density (via Canny)** which strongly correlates with grain size, combined with the vital `ppm` scaling factor.
3. **Multi-Output Regression:** Train a LightGBM regressor wrapped in `MultiOutputRegressor` to simultaneously predict all 11 target sieve fractions.
4. **Monotonicity Post-Processing:** Ensure the final predictions strictly obey the physical laws of a cumulative distribution (values must be between 0-100% and monotonically increase).

## 1. Import Dependencies & Set Paths

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import warnings

warnings.filterwarnings("ignore")

# Correct updated data paths
DATA_DIR = "/kaggle/input/competitions/soil-grain-size-from-photos"
TRAIN_LABELS_PATH = os.path.join(DATA_DIR, "Training_labels_without_H374.csv")
PPM_PATH = os.path.join(DATA_DIR, "ppm.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "Training-All_Photos_without_H374")
TEST_IMG_DIR = os.path.join(DATA_DIR, "Test_All_Photos")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")

## 2. Efficient Image Path Mapping

In [2]:
def build_image_map(directory):
    """Scans the directory once and builds a case-insensitive mapping of sample_id to full path."""
    img_map = {}
    if not os.path.exists(directory):
        return img_map
    for root, _, files in os.walk(directory):
        for file in files:
            # Assumes file name before extension is the sample_id (e.g., F827.jpg -> f827)
            name_idx = file.rfind(".")
            if name_idx != -1:
                sample_id = file[:name_idx].lower()
                img_map[sample_id] = os.path.join(root, file)
    return img_map


print("Mapping training and testing image paths...")
train_img_map = build_image_map(TRAIN_IMG_DIR)
test_img_map = build_image_map(TEST_IMG_DIR)
print(f"Mapped {len(train_img_map)} train images and {len(test_img_map)} test images.")

Mapping training and testing image paths...
Mapped 130 train images and 35 test images.


## 3. Feature Extraction Pipeline

In [3]:
def extract_features(df, img_map, ppm_lookup, global_mean_ppm, target_ppm=15.0):
    """Extracts features from images AFTER normalizing them to a unified physical scale."""
    features_list = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
        sample_id = str(row["sample_id"]).lower()
        img_path = img_map.get(sample_id)

        feat = {"sample_id": row["sample_id"]}

        if img_path and os.path.exists(img_path):
            img = cv2.imread(img_path)

            if img is not None:
                orig_h, orig_w = img.shape[0], img.shape[1]

                # 1. Find the exact PPM for this specific resolution
                current_ppm = ppm_lookup.get((orig_w, orig_h), global_mean_ppm)
                feat["ppm"] = current_ppm

                # 2. Dynamic Rescaling to unified physical scale
                scale_factor = target_ppm / current_ppm
                new_w = int(orig_w * scale_factor)
                new_h = int(orig_h * scale_factor)

                # Resize image so 1 pixel means the exact same physical size everywhere
                img_scaled = cv2.resize(img, (new_w, new_h))

                feat["img_height_scaled"] = img_scaled.shape[0]
                feat["img_width_scaled"] = img_scaled.shape[1]

                # Convert scaled image to RGB and Gray
                img_rgb = cv2.cvtColor(img_scaled, cv2.COLOR_BGR2RGB)
                gray = cv2.cvtColor(img_scaled, cv2.COLOR_BGR2GRAY)

                # 3. Color statistics from scaled image
                for i, col in enumerate(["R", "G", "B"]):
                    feat[f"{col}_mean"] = np.mean(img_rgb[:, :, i])
                    feat[f"{col}_std"] = np.std(img_rgb[:, :, i])

                # 4. Edge Density (Canny) - NOW ACCURATE because scale is uniform!
                blurred = cv2.GaussianBlur(gray, (5, 5), 0)
                edges = cv2.Canny(blurred, 30, 100)
                feat["edge_density"] = np.mean(edges > 0)

                # 5. Brightness quantiles
                feat["gray_mean"] = np.mean(gray)
                feat["gray_std"] = np.std(gray)
                feat["gray_p25"] = np.percentile(gray, 25)
                feat["gray_p75"] = np.percentile(gray, 75)
            else:
                feat["ppm"] = global_mean_ppm
                feat["edge_density"] = np.nan
        else:
            feat["ppm"] = global_mean_ppm
            feat["edge_density"] = np.nan

        features_list.append(feat)

    return pd.DataFrame(features_list)

## 4. Data Preparation & Engineering

In [4]:
# Load initial datasets
df_train_labels = pd.read_csv(TRAIN_LABELS_PATH)
df_sub = pd.read_csv(SAMPLE_SUB_PATH)
df_ppm = pd.read_csv(PPM_PATH)

# Create a fast lookup table: (width, height) -> ppm
ppm_lookup = df_ppm.set_index(["width", "height"])["ppm"].to_dict()
global_mean_ppm = df_ppm["ppm"].mean()

# Extract image features with proper individual scaling
print("Extracting features for Training Set...")
df_train_feats = extract_features(
    df_train_labels, train_img_map, ppm_lookup, global_mean_ppm
)

print("Extracting features for Test Set...")
df_test_feats = extract_features(df_sub, test_img_map, ppm_lookup, global_mean_ppm)

# Define target and feature columns
target_cols = [col for col in df_train_labels.columns if col != "sample_id"]

# Select all engineered features except the ID
feature_cols = [col for col in df_train_feats.columns if col != "sample_id"]

# Prepare X, Y and X_test matrices
X = df_train_feats[feature_cols].fillna(0)
Y = df_train_labels[target_cols].values
X_test = df_test_feats[feature_cols].fillna(0)

print(f"\n➔ Ready! Training features shape: {X.shape}")
print(f"➔ Feature columns used: {list(X.columns)}")

Extracting features for Training Set...


Extracting features: 100%|██████████| 25/25 [00:00<00:00, 12171.51it/s]


Extracting features for Test Set...


Extracting features: 100%|██████████| 10/10 [00:00<00:00, 8752.72it/s]


➔ Ready! Training features shape: (25, 2)
➔ Feature columns used: ['ppm', 'edge_density']


## 5. Multi-Output Training with K-Fold Validation

In [5]:
# Setup LightGBM Regressor Hyperparameters
lgb_params = {
    "objective": "regression_l1",  # MAE optimization
    "metric": "mae",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": 6,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbose": -1,
    "random_state": 42,
}

# Wrap in MultiOutputRegressor to support the 11 sieve dimensions simultaneously
base_model = lgb.LGBMRegressor(**lgb_params)
model = MultiOutputRegressor(base_model)

# Cross-Validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(Y.shape)
test_preds = np.zeros((len(X_test), Y.shape[1]))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, Y)):
    X_train, Y_train = X.iloc[train_idx], Y[train_idx]
    X_val, Y_val = X.iloc[val_idx], Y[val_idx]

    # Fit the multi-output wrapper
    model.fit(X_train, Y_train)

    # Predict validation fold
    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred

    # Predict Test set (accumulating fold predictions)
    test_preds += model.predict(X_test) / kf.n_splits

    fold_mae = mean_absolute_error(Y_val, val_pred)
    print(f"Fold {fold+1} Overall MAE: {fold_mae:.4f}")

overall_oof_mae = mean_absolute_error(Y, oof_preds)
print(f"\n➔ Out-of-Fold Grand MAE: {overall_oof_mae:.4f}")

Fold 1 Overall MAE: 19.4684
Fold 2 Overall MAE: 14.4785
Fold 3 Overall MAE: 14.1955
Fold 4 Overall MAE: 21.3568
Fold 5 Overall MAE: 18.0936

➔ Out-of-Fold Grand MAE: 17.5186


## 6. Post-Processing & Submission Generation

### 💡 Post-Processing for Cumulative Distributions
Because our targets represent **cumulative percentages passing through sieves**, two physical constraints must hold true:
1. Predictions must be bounded between `0.0` and `100.0`.
2. As the sieve size increases, the percentage **must strictly increase or stay the same** (monotonicity).

We will clip values and enforce sorting across the target matrix rows. This is a classic distribution regression post-processing trick.

In [6]:
def enforce_constraints(preds):
    """Enforces bounding limits and monotonic non-decreasing progression across the sieve rows."""
    # Clip between 0 and 100
    preds_clipped = np.clip(preds, 0.0, 100.0)

    # Sort each row independently to guarantee cumulative monotonicity
    preds_sorted = np.sort(preds_clipped, axis=1)

    return preds_sorted


# Apply the Kaggle distribution hack
final_test_preds = enforce_constraints(test_preds)

# Create submission DataFrame
submission = pd.DataFrame(final_test_preds, columns=target_cols)
submission.insert(0, "sample_id", df_sub["sample_id"])

# Save file
submission.to_csv("submission.csv", index=False)
print("Submission successfully saved!")
display(submission.head())

Submission successfully saved!


,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,HPC_Airbus BS6-3,1.26335,3.5909,8.9331,18.43059,31.463699,48.398669,60.89809,82.575908,95.802567,100.0,100.0
1,HPC_Audorfring,1.26335,3.5909,8.9331,18.43059,31.463699,48.398669,60.89809,82.575908,95.802567,100.0,100.0
2,HPC_Muenster_BS6_9_0-10m,1.26335,3.5909,8.9331,18.43059,31.463699,48.398669,60.89809,82.575908,95.802567,100.0,100.0
3,HPC_Airbus BS10-4bis7,1.26335,3.5909,8.9331,18.43059,31.463699,48.398669,60.89809,82.575908,95.802567,100.0,100.0
4,HPC_Airbus BS12-5bis8,1.26335,3.5909,8.9331,18.43059,31.463699,48.398669,60.89809,82.575908,95.802567,100.0,100.0
